# ForexGuard — Synthetic Dataset EDA

This notebook performs quick, interview-friendly EDA on the **synthetic event stream** used by ForexGuard.

It regenerates the dataset using the repo's generator (`data/generate_synthetic.py`) so results are reproducible via a fixed seed.

**Goals**
- Validate the dataset shape (event types, date range, anomaly rate)
- Visualize distributions tied to our feature engineering (time deltas, IP/device sharing, volume spikes)
- Produce a small set of plots that explain *why* the model features make sense


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data.generate_synthetic import generate_dataset, get_anomaly_statistics

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 5)
pd.set_option('display.max_columns', 200)


In [ ]:
# Regenerate dataset deterministically
NUM_USERS = 500
NUM_EVENTS = 50_000
ANOMALY_RATE = 0.05
SEED = 42

df = generate_dataset(num_users=NUM_USERS, num_events=NUM_EVENTS, anomaly_rate=ANOMALY_RATE, seed=SEED)
stats = get_anomaly_statistics(df)
stats


In [ ]:
df.head()


## 1) Event volume over time + anomaly rate
This approximates how a broker's event stream ebbs/flows and lets us sanity-check that anomalies are not concentrated in a single time bucket.

In [ ]:
ts = df.set_index('timestamp').sort_index()
daily = ts.resample('D').agg(events=('event_id','count'), anomaly_rate=('is_anomaly','mean'))

fig, ax1 = plt.subplots()
ax1.plot(daily.index, daily['events'], color='steelblue', label='Events/day')
ax1.set_title('Events per day + anomaly rate')
ax1.set_ylabel('Events/day')

ax2 = ax1.twinx()
ax2.plot(daily.index, daily['anomaly_rate']*100, color='darkred', alpha=0.8, label='Anomaly rate (%)')
ax2.set_ylabel('Anomaly rate (%)')

lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc='upper right')
plt.show()


## 2) Event types (overall + anomalies)
We want a realistic mix (login/trade/payments/portal actions) and see which event types are over-represented in anomalies.

In [ ]:
event_counts = df['event_type'].value_counts().head(20)
plt.figure(figsize=(11, 6))
sns.barplot(x=event_counts.values, y=event_counts.index, color='slateblue')
plt.title('Top event types (count)')
plt.xlabel('count')
plt.ylabel('event_type')
plt.show()

anom_counts = df[df['is_anomaly'] == True]['event_type'].value_counts().head(20)
plt.figure(figsize=(11, 6))
sns.barplot(x=anom_counts.values, y=anom_counts.index, color='indianred')
plt.title('Top event types among anomalies (count)')
plt.xlabel('count')
plt.ylabel('event_type')
plt.show()


## 3) Activity per user (long-tail)
Fraud systems must handle long-tail user activity. We visualize events/user and anomalies/user.

In [ ]:
events_per_user = df.groupby('user_id')['event_id'].count().rename('events')
anom_per_user = df[df['is_anomaly'] == True].groupby('user_id')['event_id'].count().rename('anomalies')
user_summary = pd.concat([events_per_user, anom_per_user], axis=1).fillna(0).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(user_summary['events'], bins=40, ax=axes[0], color='steelblue')
axes[0].set_title('Events per user')
axes[0].set_xlabel('events')

sns.histplot(user_summary['anomalies'], bins=40, ax=axes[1], color='darkred')
axes[1].set_title('Anomalies per user')
axes[1].set_xlabel('anomalies')
plt.tight_layout()
plt.show()

user_summary.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T


## 4) Inter-event time (bot-like vs human-like cadence)
We compute per-user time deltas between consecutive events. Bot-like navigation often shows very small deltas.

In [ ]:
df_sorted = df.sort_values(['user_id', 'timestamp']).copy()
df_sorted['delta_s'] = df_sorted.groupby('user_id')['timestamp'].diff().dt.total_seconds()
deltas = df_sorted['delta_s'].dropna()

plt.figure(figsize=(11, 4))
sns.histplot(np.log1p(deltas), bins=60, color='teal')
plt.title('log(1 + inter-event delta seconds)')
plt.xlabel('log1p(delta_s)')
plt.show()

deltas.describe(percentiles=[0.5, 0.9, 0.95, 0.99])


## 5) IP / device sharing (hub behavior)
We look at how many distinct users share the same IP or device string. In the product, we track this in rolling windows; here we visualize global sharing to confirm the signal exists.

In [ ]:
def plot_hub_counts(col: str, top_n: int = 15):
    if col not in df.columns:
        print(f"Column {col} not found")
        return
    hub = df.groupby(col)['user_id'].nunique().sort_values(ascending=False).head(top_n)
    plt.figure(figsize=(11, 5))
    sns.barplot(x=hub.values, y=hub.index.astype(str), color='gray')
    plt.title(f'Top {top_n} {col} values by number of distinct users')
    plt.xlabel('distinct users')
    plt.ylabel(col)
    plt.show()
    return hub

ip_hub = plot_hub_counts('ip_address', top_n=15)
device_hub = plot_hub_counts('device', top_n=15)


## 6) Trading distributions (lot size, margin used)
We compare normal vs anomalous distributions. Large shifts justify deviation features like z-scores.

In [ ]:
trades = df[df['event_type'] == 'trade'].copy()
if len(trades) == 0:
    print('No trade events found')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    if 'lot_size' in trades.columns:
        sns.kdeplot(data=trades, x='lot_size', hue='is_anomaly', common_norm=False, ax=axes[0])
        axes[0].set_title('lot_size distribution (trade)')
    if 'margin_used' in trades.columns:
        sns.kdeplot(data=trades, x='margin_used', hue='is_anomaly', common_norm=False, ax=axes[1])
        axes[1].set_title('margin_used distribution (trade)')
    plt.tight_layout()
    plt.show()


## 7) Portal activity (page views, support tickets, document uploads)
These support the client-portal anomaly signals (rapid navigation, ticket spikes, upload spam/failures).

In [ ]:
portal_types = ['page_view', 'support_ticket', 'document_upload', 'profile_update', 'kyc_change']
portal = df[df['event_type'].isin(portal_types)].copy()
print('Portal events:', len(portal))

if len(portal) > 0:
    counts = portal['event_type'].value_counts()
    plt.figure(figsize=(9, 4))
    sns.barplot(x=counts.index, y=counts.values, color='mediumpurple')
    plt.title('Portal event counts')
    plt.xticks(rotation=30, ha='right')
    plt.ylabel('count')
    plt.show()

    # Document upload success rate (if present)
    if 'upload_success' in portal.columns and portal['event_type'].eq('document_upload').any():
        du = portal[portal['event_type'] == 'document_upload'].copy()
        du['upload_success'] = du['upload_success'].fillna(False).astype(bool)
        rate = du['upload_success'].mean()
        print(f'Document upload success rate: {rate:.2%}')
        sns.countplot(data=du, x='upload_success')
        plt.title('Document upload success/failure counts')
        plt.show()


## Key takeaways (what to say in interviews)
- **Long-tail activity per user** supports per-user baselines + rolling features
- **Inter-event deltas** separate bot-like bursts from normal cadence
- **IP/device sharing** validates hub/collusion-style signals
- **Trade distribution shifts** justify deviation/z-score features and outlier-focused models like Isolation Forest
